In [26]:
import numpy as np
import cv2
from scipy.ndimage import gaussian_filter

def UIF_main(D, R):
    # D is the enhancement image; R is the raw image.

    Dg = np.double(cv2.cvtColor(D, cv2.COLOR_BGR2GRAY))
    Rg = np.double(cv2.cvtColor(R, cv2.COLOR_BGR2GRAY))
    D = np.double(D)
    R = np.double(R)

    # Basic SETUP OF FRSIM
    d = np.diff(np.array([np.min(D), np.max(D)]))
    C = np.array([(0.01 * d) ** 2, (0.01 * d) ** 2 / 2, (0.03 * d) ** 2 / 2])
    a = 0.8
    b = 0.2

    # MSCN
    MSCN_window = cv2.getGaussianKernel(7, 7 / 6)
    MSCN_window = MSCN_window * MSCN_window.T
    MSCN_window = MSCN_window / np.sum(MSCN_window)

    mu = cv2.filter2D(Dg, -1, MSCN_window, borderType=cv2.BORDER_REPLICATE)  # Enhancement image's MSCN.
    mu_sq = mu * mu
    sigma = np.sqrt(np.abs(cv2.filter2D(Dg * Dg, -1, MSCN_window, borderType=cv2.BORDER_REPLICATE) - mu_sq))
    D_MSCN = (Dg - mu) / (sigma + 1)

    mu1 = cv2.filter2D(Rg, -1, MSCN_window, borderType=cv2.BORDER_REPLICATE)  # Raw image's MSCN.
    mu_sq1 = mu1 * mu1
    sigma1 = np.sqrt(np.abs(cv2.filter2D(Rg * Rg, -1, MSCN_window, borderType=cv2.BORDER_REPLICATE) - mu_sq1))
    R_MSCN = (Rg - mu1) / (sigma1 + 1)

    # feature extraction and calculation   
    # Naturalness Index
    mc_mscn = max(np.var(D_MSCN), np.var(R_MSCN))
    w_mscn = mc_mscn / np.sum(mc_mscn)
    CM = C[0]
    SM = (2 * D_MSCN * R_MSCN + CM) / (D_MSCN * D_MSCN + R_MSCN * R_MSCN + CM)
    mean_SM = np.mean(SM)

    # Sharpness Index
    D_gradient = grad(D)  # Placeholder for gradient function
    R_gradient = grad(R)  # Placeholder for gradient function
    CG = C[1]
    SG = list((2 * D_gradient * R_gradient + CG) / (D_gradient * D_gradient + R_gradient * R_gradient + CG))
    mean_SG = np.mean(SG)

    # Structure Index
    CS = C[2]
    SS = (2 * sigma * sigma1 + CS) / (sigma ** 2 + sigma1 ** 2 + CS)
    mean_SS = np.mean(SS)

    # pooling
    UIFscore = mean_SS ** a * mean_SG ** b * mean_SM
    return UIFscore



In [27]:
import numpy as np
from scipy.ndimage import convolve

def grad(Ig):
    Ig = np.double(Ig[:, :, 0])
    Ig = Ig / 255 + np.finfo(float).eps
    hy = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]])  # Sobel filter
    hx = hy.T
    Iy = convolve(Ig, hy, mode='nearest')
    Ix = convolve(Ig, hx, mode='nearest')
    gradimg = np.sqrt(Ix**2 + Iy**2)
    return gradimg




In [28]:
import numpy as np
from scipy.signal import convolve2d
from scipy.ndimage import gaussian_filter

def VSI(image1, image2):
    # Constants
    constForVS = 1.27
    constForGM = 386
    constForChrom = 130
    alpha = 0.40
    lambda_ = 0.020
    sigmaF = 1.34
    omega0 = 0.0210
    sigmaD = 145
    sigmaC = 0.001

    # Compute the visual saliency map using SDSP (assuming SDSP function is defined elsewhere)
    saliencyMap1 = SDSP(image1, sigmaF, omega0, sigmaD, sigmaC)
    saliencyMap2 = SDSP(image2, sigmaF, omega0, sigmaD, sigmaC)

    rows, cols, _ = image1.shape

    # Transform into an opponent color space
    L1 = 0.06 * image1[:,:,0] + 0.63 * image1[:,:,1] + 0.27 * image1[:,:,2]
    L2 = 0.06 * image2[:,:,0] + 0.63 * image2[:,:,1] + 0.27 * image2[:,:,2]
    M1 = 0.30 * image1[:,:,0] + 0.04 * image1[:,:,1] - 0.35 * image1[:,:,2]
    M2 = 0.30 * image2[:,:,0] + 0.04 * image2[:,:,1] - 0.35 * image2[:,:,2]
    N1 = 0.34 * image1[:,:,0] - 0.60 * image1[:,:,1] + 0.17 * image1[:,:,2]
    N2 = 0.34 * image2[:,:,0] - 0.60 * image2[:,:,1] + 0.17 * image2[:,:,2]

    # Downsample the image
    minDimension = min(rows, cols)
    F = max(1, round(minDimension / 256))
    aveKernel = np.ones((F, F)) / (F * F)

    M1 = convolve2d(M1, aveKernel, mode='same')[::F, ::F]
    M2 = convolve2d(M2, aveKernel, mode='same')[::F, ::F]
    N1 = convolve2d(N1, aveKernel, mode='same')[::F, ::F]
    N2 = convolve2d(N2, aveKernel, mode='same')[::F, ::F]
    L1 = convolve2d(L1, aveKernel, mode='same')[::F, ::F]
    L2 = convolve2d(L2, aveKernel, mode='same')[::F, ::F]
    saliencyMap1 = convolve2d(saliencyMap1, aveKernel, mode='same')[::F, ::F]
    saliencyMap2 = convolve2d(saliencyMap2, aveKernel, mode='same')[::F, ::F]

    # Calculate the gradient map
    dx = np.array([[3, 0, -3], [10, 0, -10], [3, 0, -3]]) / 16
    dy = np.array([[3, 10, 3], [0, 0, 0], [-3, -10, -3]]) / 16

    IxL1 = convolve2d(L1, dx, mode='same')
    IyL1 = convolve2d(L1, dy, mode='same')
    gradientMap1 = np.sqrt(IxL1**2 + IyL1**2)

    IxL2 = convolve2d(L2, dx, mode='same')
    IyL2 = convolve2d(L2, dy, mode='same')
    gradientMap2 = np.sqrt(IxL2**2 + IyL2**2)

    # Calculate the VSI
    VSSimMatrix = (2 * saliencyMap1 * saliencyMap2 + constForVS) / (saliencyMap1**2 + saliencyMap2**2 + constForVS)
    gradientSimMatrix = (2 * gradientMap1 * gradientMap2 + constForGM) / (gradientMap1**2 + gradientMap2**2 + constForGM)

    weight = np.maximum(saliencyMap1, saliencyMap2)

    ISimMatrix = (2 * M1 * M2 + constForChrom) / (M1**2 + M2**2 + constForChrom)
    QSimMatrix = (2 * N1 * N2 + constForChrom) / (N1**2 + N2**2 + constForChrom)

    SimMatrixC = (gradientSimMatrix ** alpha) * VSSimMatrix * np.real((ISimMatrix * QSimMatrix) ** lambda_) * weight
    sim = np.sum(SimMatrixC) / np.sum(weight)

    return sim



In [29]:
import numpy as np
from scipy import fftpack
from skimage import color, transform

def SDSP(image, sigmaF, omega0, sigmaD, sigmaC):
    """
    SDSP algorithm for salient region detection from a given image.
    
    Input:
    - image: an uint8 RGB image with dynamic range [0, 255] for each channel
    - sigmaF, omega0, sigmaD, sigmaC: algorithm parameters
    
    Output:
    - VSMap: the visual saliency map extracted by the SDSP algorithm.
    Data range for VSMap is [0, 1]. It can be regarded as a common gray-scale image.
    """
    
    # Convert the image into LAB color space
    oriRows, oriCols = image.shape[:2]
    image = image.astype(np.float64)
    dsImage = transform.resize(image, (256, 256), mode='reflect')
    lab = color.rgb2lab(dsImage)
    
    LChannel, AChannel, BChannel = lab[:,:,0], lab[:,:,1], lab[:,:,2]
    
    LFFT = fftpack.fft2(LChannel)
    AFFT = fftpack.fft2(AChannel)
    BFFT = fftpack.fft2(BChannel)
    
    rows, cols = dsImage.shape[:2]
    LG = logGabor(rows, cols, omega0, sigmaF)
    FinalLResult = np.real(fftpack.ifft2(LFFT * LG))
    FinalAResult = np.real(fftpack.ifft2(AFFT * LG))
    FinalBResult = np.real(fftpack.ifft2(BFFT * LG))
    
    SFMap = np.sqrt(FinalLResult**2 + FinalAResult**2 + FinalBResult**2)
    
    # The central areas will have a bias towards attention
    y, x = np.ogrid[:rows, :cols]
    centerY, centerX = rows / 2, cols / 2
    SDMap = np.exp(-((y - centerY)**2 + (x - centerX)**2) / (2 * sigmaD**2))
    
    # Warm colors have a bias towards attention
    normalizedA = (AChannel - AChannel.min()) / (AChannel.max() - AChannel.min())
    normalizedB = (BChannel - BChannel.min()) / (BChannel.max() - BChannel.min())
    
    labDistSquare = normalizedA**2 + normalizedB**2
    SCMap = 1 - np.exp(-labDistSquare / (sigmaC**2))
    
    VSMap = SFMap * SDMap * SCMap
    
    VSMap = transform.resize(VSMap, (oriRows, oriCols), mode='reflect')
    VSMap = (VSMap - VSMap.min()) / (VSMap.max() - VSMap.min())
    
    return VSMap




In [30]:
import numpy as np

def RGB2Lab(image):
    image = image.astype(np.float64)
    normalizedR = image[:,:,0] / 255
    normalizedG = image[:,:,1] / 255
    normalizedB = image[:,:,2] / 255

    RSmallerOrEqualto4045 = normalizedR <= 0.04045
    RGreaterThan4045 = ~RSmallerOrEqualto4045
    tmpR = (normalizedR / 12.92) * RSmallerOrEqualto4045
    tmpR += np.power((normalizedR + 0.055)/1.055, 2.4) * RGreaterThan4045

    GSmallerOrEqualto4045 = normalizedG <= 0.04045
    GGreaterThan4045 = ~GSmallerOrEqualto4045
    tmpG = (normalizedG / 12.92) * GSmallerOrEqualto4045
    tmpG += np.power((normalizedG + 0.055)/1.055, 2.4) * GGreaterThan4045

    BSmallerOrEqualto4045 = normalizedB <= 0.04045
    BGreaterThan4045 = ~BSmallerOrEqualto4045
    tmpB = (normalizedB / 12.92) * BSmallerOrEqualto4045
    tmpB += np.power((normalizedB + 0.055)/1.055, 2.4) * BGreaterThan4045

    X = tmpR*0.4124564 + tmpG*0.3575761 + tmpB*0.1804375
    Y = tmpR*0.2126729 + tmpG*0.7151522 + tmpB*0.0721750
    Z = tmpR*0.0193339 + tmpG*0.1191920 + tmpB*0.9503041

    epsilon = 0.008856  # actual CIE standard
    kappa   = 903.3     # actual CIE standard
     
    Xr = 0.9642  # reference white D50
    Yr = 1.0     # reference white
    Zr = 0.8251  # reference white

    xr = X/Xr
    yr = Y/Yr
    zr = Z/Zr

    xrGreaterThanEpsilon = xr > epsilon
    xrSmallerOrEqualtoEpsilon = ~xrGreaterThanEpsilon
    fx = np.power(xr, 1.0/3.0) * xrGreaterThanEpsilon
    fx += (kappa*xr + 16.0)/116.0 * xrSmallerOrEqualtoEpsilon

    yrGreaterThanEpsilon = yr > epsilon
    yrSmallerOrEqualtoEpsilon = ~yrGreaterThanEpsilon
    fy = np.power(yr, 1.0/3.0) * yrGreaterThanEpsilon
    fy += (kappa*yr + 16.0)/116.0 * yrSmallerOrEqualtoEpsilon

    zrGreaterThanEpsilon = zr > epsilon
    zrSmallerOrEqualtoEpsilon = ~zrGreaterThanEpsilon
    fz = np.power(zr, 1.0/3.0) * zrGreaterThanEpsilon
    fz += (kappa*zr + 16.0)/116.0 * zrSmallerOrEqualtoEpsilon

    rows, cols = image.shape[:2]
    labImage = np.zeros((rows, cols, 3))
    labImage[:,:,0] = 116.0 * fy - 16.0
    labImage[:,:,1] = 500.0 * (fx - fy)
    labImage[:,:,2] = 200.0 * (fy - fz)
    
    return labImage



In [31]:
import numpy as np
from scipy import signal

def log_gabor(rows, cols, omega0, sigma_f):
    u1, u2 = np.meshgrid((np.arange(cols) - (cols // 2 + 1)) / (cols - cols % 2),
                         (np.arange(rows) - (rows // 2 + 1)) / (rows - rows % 2))
    
    mask = np.ones((rows, cols))
    mask[u1**2 + u2**2 > 0.25] = 0
    
    u1 *= mask
    u2 *= mask
    
    u1 = np.fft.ifftshift(u1)
    u2 = np.fft.ifftshift(u2)
    
    radius = np.sqrt(u1**2 + u2**2)
    radius[0, 0] = 1
    
    lg = np.exp(-(np.log(radius / omega0)**2) / (2 * sigma_f**2))
    lg[0, 0] = 0
    
    return lg



In [32]:
import numpy as np
import cv2

constForVS = 1.27  # fixed
constForGM = 386   # fixed
constForChrom = 130  # fixed
alpha = 0.40  # fixed
lambda_ = 0.020  # fixed
sigmaF = 1.34  # fixed donot change
omega0 = 0.0210  # fixed
sigmaD = 145  # fixed
sigmaC = 0.001  # fixed

#img1 = cv2.imread('ori.bmp')
#img2 = cv2.imread('enh.bmp')
img1 = iio.imread("g4g.png")
img2 = iio.imread("g4g.png")
UIFscore = UIF_main(img1, img2)
print(UIFscore)
# UIFscore = simple_UIF(img1, img2)



0.4257812198014837
